# Tutorial and Data Extraction

Basic usage notes and the old data-extraction scratch work, grouped as the entry notebook.

All cells are intentionally unexecuted; run sections selectively from top to bottom.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / 'core').exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)


## `notebooks/tutorial.ipynb`

Original tutorial notebook.


In [ ]:
import numpy as np
from scipy.sparse import lil_matrix
from utils.generic_model import GenericModel

! pip install -r requirements.txt
! conda install -c marmote -c conda-forge marmote


### Model writing


In [ ]:
# Steps:
# - Write an MDP with the following code format.
# - Create a file in "/models" and add it to the comparison. Add the authors to the top of the file if needed.

class Model(GenericModel):
    """
    Tutorial template: how to encode a finite MDP.
    Conventions:
      - Each transition T[a, s, :] row (action a and initial state s fixed) should sum to 1 (row-stochastic).
    """
    def __init__(self, state_dim: int, action_dim: int):
        """
        Expected attributes:
        - self.state_dim: int,
        - self.action_dim: int,
        - self.name : str.
        """
        self.state_dim = int(state_dim) # Or a fixed parameter.
        self.action_dim = int(action_dim) # Or a fixed parameter.

        self.name = f"{self.state_dim}_{self.action_dim}_name"

    def _build_model(self) -> None:
        """
        Expected attributes:
        - self.reward_matrix: ndarray, shape (S, A)
                reward_matrix[s, a] = R(s, a)
        - self.transition_matrix: list[csr_matrix], length A, each shape (S, S)
                transition_matrix[a][s, s_next] = P(s_next | s, a)
        """
        S, A = self.state_dim, self.action_dim

        # Reward matrix definition.
        self.reward_matrix = -1.0 * np.ones((S, A), dtype=np.float64)

        P_lil = [lil_matrix((S, S), dtype=np.float64) for _ in range(A)]
        for s in range(S):
            for a in range(A):
                s_next = s
                prob = 1.0
                P_lil[a][s, s_next] = prob
        self.transition_matrix = [Pa.tocsr() for Pa in P_lil]


### Instanciation and solving


In [ ]:
from solvers.marmote_vi import Solver as MarmoteVi

model = Model(100, 10)
model.create_model()

solver = MarmoteVi(model, 0.9, 1e-2)
solver.run()

print(f"Marmote Value Iteration Runtime: {solver.runtime}")


### Comparison of solvers


In [ ]:
from utils.data_management import solve

state_dim = 1000

solvers = ["mdptoolbox_vi", "mdptoolbox_pim", "marmote_vi", "marmote_mpi"] # Name of the python files in folder "solvers". Complete if necessary.
models = ['rooms', 'taxi'] # Name of the python files in folder "models". Complete if necessary.

results = {}

for solver_name in solvers:
    for model_name in models:
        model, solver, mean, std = solve(model_name, solver_name, state_dim, repeat=5)
        print(f'For model {model.name}, solver {solver.name}, runtime mean: {mean:.3f}. std {std:.3f}.')
        results[solver_name] = mean

print()
best = min(results, key=results.get)
print(f'Best solver: {best} with runtime {results[best]:.3f}.')


## `notebooks/research/data_extraction.ipynb`

Research notebook for extracting and inspecting stored experiment data.


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

RESULTS_PATH = os.path.join(os.getcwd(), 'results')

def excel_path(file: str):
    return os.path.join(RESULTS_PATH, file)

results_files = [excel_path(file) for file in os.listdir(RESULTS_PATH)]

def read_result_file(file):
    return pd.read_csv(file).drop(['Unnamed: 0'], axis=1)

def select_rows(df, discount=None, precision=None, min_state_dim=None, max_state_dim=None):
    if discount is not None:
        df = df[df['discount'] == discount]
    if precision is not None:
        df = df[df['final_precision'] == precision]
    if min_state_dim is not None:
        df = df[df['state_dim'] >= min_state_dim]
    if max_state_dim is not None:
        df = df[df['state_dim'] <= max_state_dim]
    return df

for i, elem in enumerate(results_files):
    print("{} {}".format(i, elem))


In [ ]:
model = 11
df = read_result_file(results_files[model])
df = select_rows(df=df, discount=0.99, precision=1e-2, min_state_dim=100, max_state_dim=10000)
df


### Aggregation comparison


In [ ]:
adversarial_results = df[df['solver_name'].apply(lambda x : ('personal' in x) or ('bertsekas' in x) or ('chen' in x))]
assert len(adversarial_results) == 4
adversarial_results


In [ ]:
aggregated_results = df[df['solver_name'].apply(lambda x : 'aggregated' in x)]
assert len(aggregated_results) == 3
aggregated_results


In [ ]:
if aggregated_results['runtime'].min() < adversarial_results['runtime'].min():
    print('Success of aggregation.')
else:
    print("Failed")


## `notebooks/research/log_update.ipynb`

Small log/update notebook retained as a historical notes section.


In [ ]:
from gurobipy import norm
import numpy as np

from models.rooms import Model
from utils.generic_model import GenericModel

model = Model(100, 4)
model.create_model(normalize_reward=True)
model._model_to_numpy()


In [ ]:
model.reward_matrix += 1


In [ ]:
def log_bellman_operator_numpy(model:GenericModel, log_value:np.ndarray, discount:float) -> np.ndarray:
    value = np.exp(log_value)
    q_value = model.reward_matrix + discount * model.transition_matrix.dot(value).T
    q_value = np.log(q_value)
    return np.max(q_value, axis=1)

log_value = np.zeros(model.state_dim)
discount = 0.9


for i in range(1000):
    log_value = log_bellman_operator_numpy(model, log_value, discount)

print(np.exp(log_value).reshape((10, 10)))
